In [1]:
import pandas as pd

path = "/Users/gres1/Downloads/Twitterhate.csv"
df = pd.read_csv(path)

df.head()

,id,label,tweet
0,1,0,@user when a father is dysfunctional and is s...
1,2,0,@user @user thanks for #lyft credit i can't us...
2,3,0,bihday your majesty
3,4,0,#model i love u take with u all the time in ...
4,5,0,factsguide: society now #motivation


In [10]:
def clean_tweet(text: str):
    text = text.lower()
    text = handle_re.sub("", text)
    text = url_re.sub("", text)
    text = text.replace("#", "")
    tokens = tt.tokenize(text)
    tokens = [
        t for t in tokens
        if len(t) > 1
        and t not in stop_words
        and t not in redundant
    ]
    return tokens

clean_text = [" ".join(toks) for toks in tokenized_tweets]
clean_text[:3]

['father dysfunctional selfish drags kids dysfunction run',
 "thanks lyft credit can't use cause don't offer wheelchair vans pdx disapointed getthanked",
 'bihday majesty']

In [12]:
tweets = df["tweet"].astype(str).tolist()
tweets[:3]

[' @user when a father is dysfunctional and is so selfish he drags his kids into his dysfunction.   #run',
 "@user @user thanks for #lyft credit i can't use cause they don't offer wheelchair vans in pdx.    #disapointed #getthanked",
 '  bihday your majesty']

In [13]:
import re
from nltk.tokenize import TweetTokenizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# tokenizer: lowercase handled by us, but this is tweet-friendly
tt = TweetTokenizer(preserve_case=False, reduce_len=True, strip_handles=False)

stop_words = set(ENGLISH_STOP_WORDS)
redundant = {"amp", "rt"}  # you can add more later

handle_re = re.compile(r"@\w+")
url_re = re.compile(r"http\S+|www\.\S+")



In [14]:
tokenized_tweets = [clean_tweet(t) for t in tweets]

tokenized_tweets[:3]

[['father', 'dysfunctional', 'selfish', 'drags', 'kids', 'dysfunction', 'run'],
 ['thanks',
  'lyft',
  'credit',
  "can't",
  'use',
  'cause',
  "don't",
  'offer',
  'wheelchair',
  'vans',
  'pdx',
  'disapointed',
  'getthanked'],
 ['bihday', 'majesty']]

In [6]:
for i in range(3):
    print("ORIGINAL:", tweets[i])
    print("TOKENS:  ", tokenized_tweets[i])
    print("---")

ORIGINAL:  @user when a father is dysfunctional and is so selfish he drags his kids into his dysfunction.   #run
TOKENS:   ['father', 'dysfunctional', 'selfish', 'drags', 'kids', 'dysfunction', 'run']
---
ORIGINAL: @user @user thanks for #lyft credit i can't use cause they don't offer wheelchair vans in pdx.    #disapointed #getthanked
TOKENS:   ['thanks', 'lyft', 'credit', "can't", 'use', 'cause', "don't", 'offer', 'wheelchair', 'vans', 'pdx', 'disapointed', 'getthanked']
---
ORIGINAL:   bihday your majesty
TOKENS:   ['bihday', 'majesty']
---


In [15]:
from collections import Counter

all_terms = []
for tweet_tokens in tokenized_tweets:
    all_terms.extend(tweet_tokens)

top10 = Counter(all_terms).most_common(10)
top10

[('...', 2808),
 ('love', 2680),
 ('day', 2273),
 ('happy', 1683),
 ('just', 1365),
 ('time', 1128),
 ('life', 1103),
 ('like', 1097),
 ("it's", 1057),
 ("i'm", 1018)]

In [16]:
clean_text = [" ".join(toks) for toks in tokenized_tweets]

In [17]:
X = clean_text
y = df["label"].values

In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [21]:
import numpy as np
print("Train % hate:", np.mean(y_train))
print("Test  % hate:", np.mean(y_test))
print("Train size:", len(X_train), "Test size:", len(X_test))

Train % hate: 0.07016308811451367
Test  % hate: 0.07007664633192555
Train size: 25569 Test size: 6393


In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

X_train_vec = tfidf.fit_transform(X_train)
X_test_vec  = tfidf.transform(X_test)

X_train_vec.shape, X_test_vec.shape

((25569, 5000), (6393, 5000))

In [24]:
from sklearn.linear_model import LogisticRegression

lr_default = LogisticRegression()          
lr_default.fit(X_train_vec, y_train)      

y_pred_train = lr_default.predict(X_train_vec)
y_pred_test  = lr_default.predict(X_test_vec)

In [25]:
from sklearn.metrics import accuracy_score, recall_score, f1_score

train_acc = accuracy_score(y_train, y_pred_train)
train_recall = recall_score(y_train, y_pred_train)   # recall for class 1 (hate)
train_f1 = f1_score(y_train, y_pred_train)

print("TRAIN accuracy:", train_acc)
print("TRAIN recall:  ", train_recall)
print("TRAIN f1:      ", train_f1)

TRAIN accuracy: 0.9552974304822246
TRAIN recall:   0.3862876254180602
TRAIN f1:       0.5480427046263345


In [26]:
lr_balanced = LogisticRegression(class_weight="balanced")
lr_balanced.fit(X_train_vec, y_train)

y_pred_train_bal = lr_balanced.predict(X_train_vec)

train_acc_bal = accuracy_score(y_train, y_pred_train_bal)
train_recall_bal = recall_score(y_train, y_pred_train_bal)
train_f1_bal = f1_score(y_train, y_pred_train_bal)

print("BALANCED TRAIN accuracy:", train_acc_bal)
print("BALANCED TRAIN recall:  ", train_recall_bal)
print("BALANCED TRAIN f1:      ", train_f1_bal)

BALANCED TRAIN accuracy: 0.9461457233368532
BALANCED TRAIN recall:   0.9693422519509476
BALANCED TRAIN f1:       0.7163748712667353


In [27]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)

lr = LogisticRegression(class_weight="balanced", solver="liblinear")

param_grid = {
    "C": [0.01, 0.1, 1, 10],
    "penalty": ["l1", "l2"]
}

grid = GridSearchCV(
    estimator=lr,
    param_grid=param_grid,
    scoring="recall",     # we optimize for recall of class 1
    cv=cv,
    refit=True
)

grid.fit(X_train_vec, y_train)

print("Best params:", grid.best_params_)
print("Best CV recall:", grid.best_score_)

Best params: {'C': 1, 'penalty': 'l2'}
Best CV recall: 0.7803762826121541


In [28]:
best_model = grid.best_estimator_

y_pred_train_best = best_model.predict(X_train_vec)
train_acc_best = accuracy_score(y_train, y_pred_train_best)
train_recall_best = recall_score(y_train, y_pred_train_best)
train_f1_best = f1_score(y_train, y_pred_train_best)

print("BEST TRAIN accuracy:", train_acc_best)
print("BEST TRAIN recall:  ", train_recall_best)
print("BEST TRAIN f1:      ", train_f1_best)

BEST TRAIN accuracy: 0.9463021627752356
BEST TRAIN recall:   0.9693422519509476
BEST TRAIN f1:       0.7169655741084312


In [29]:
from sklearn.metrics import recall_score, f1_score, accuracy_score

print("Best parameters:", grid.best_params_)

best_model = grid.best_estimator_
y_pred_test_best = best_model.predict(X_test_vec)

print("TEST accuracy:", accuracy_score(y_test, y_pred_test_best))
print("TEST recall (hate=1):", recall_score(y_test, y_pred_test_best))
print("TEST f1 (hate=1):", f1_score(y_test, y_pred_test_best))

Best parameters: {'C': 1, 'penalty': 'l2'}
TEST accuracy: 0.9227279837322071
TEST recall (hate=1): 0.7879464285714286
TEST f1 (hate=1): 0.5883333333333334
